<a href="https://colab.research.google.com/github/ileeee-sh/SeSAC_29CM_Project/blob/main/PyKoSpacing_x_SOYNLP_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 29CM 앱 리뷰 한국어 전처리 파이프라인
1. **PyKoSpacing** - 띄어쓰기 보정
2. **SOYNLP emoticon_normalize** - 반복 이모티콘 정규화 (ㅋㅋㅋㅋ → ㅋㅋ)
3. **SOYNLP LTokenizer** - 토큰화
4. **emoji 라이브러리** - 이모지 별도 추출


In [ ]:
# !pip install git+https://github.com/haven-jeon/PyKoSpacing.git --quiet
!pip install soynlp emoji --quiet

In [2]:
import pandas as pd
import emoji
from pykospacing import Spacing
from soynlp.normalizer import emoticon_normalize
from soynlp.tokenizer import LTokenizer

In [3]:
from google.colab import files

uploaded = files.upload()  # 파일 업로드 창 실행
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)
print(f'파일 로드 완료: {df.shape[0]}행 × {df.shape[1]}열')
print(f'컬럼 목록: {list(df.columns)}')
df.head()

Saving 29cm_app_reviews.csv to 29cm_app_reviews.csv
파일 로드 완료: 1919행 × 4열
컬럼 목록: ['reviewId', 'content', 'score', 'thumbsUpCount']


,reviewId,content,score,thumbsUpCount
0,f1d2e09a-d2a6-4c97-8375-d97330b48f3e,굿,5,0
1,bffa904a-08e9-4f20-b426-fb2bf98ba4c8,할인 쿠폰이 막상 내가 사고싶은 물건엔 적용되지 않음.,1,1
2,aa3379c2-77ac-43b9-a0dd-a7979b613744,배송이 빠르고 실시간 배송정보가 알림으로 떠서 아주 만족스럽습니다.,5,0
3,4491a225-0f92-4463-91b9-969e075996a4,쇼핑하기 편리해요 쿠폰 적용도 잘되고 다양한 브랜드가 많이 있어서 쇼핑몰을 가지 않...,5,0
4,3f598513-9067-4b62-b2b8-7720d25961f4,여러스타일의 브랜드를 같이 볼수있어서 너무 좋아요~^^♡,5,0


In [4]:
# PyKoSpacing 초기화
spacing = Spacing()

In [5]:
def extract_emojis(text: str) -> list:
    """이모지만 추출하여 리스트로 반환"""
    return [char for char in text if char in emoji.EMOJI_DATA]


def remove_emojis(text: str) -> str:
    """텍스트에서 이모지 제거 (PyKoSpacing/SOYNLP 처리 전 분리용)"""
    return ''.join(char for char in text if char not in emoji.EMOJI_DATA)


def preprocess_review(text: str, spacing_model, tokenizer) -> dict:
    """
    단일 리뷰 텍스트 전처리 파이프라인

    Returns:
        dict: {
            'emojis'          : 원문에서 추출한 이모지 리스트
            'spaced'          : PyKoSpacing 띄어쓰기 보정 결과
            'normalized'      : emoticon_normalize 결과
            'tokens'          : LTokenizer 토큰 리스트
        }
    """
    if not isinstance(text, str) or text.strip() == '':
        return {
            'emojis': [],
            'spaced': text,
            'normalized': text,
            'tokens': []
        }

    # Step 1: 이모지 추출 (원문 기준)
    emojis = extract_emojis(text)

    # Step 2: 이모지 제거 후 PyKoSpacing 띄어쓰기 보정
    text_no_emoji = remove_emojis(text).strip()
    try:
        spaced = spacing_model(text_no_emoji) if text_no_emoji else text_no_emoji
    except Exception:
        spaced = text_no_emoji

    # Step 3: SOYNLP emoticon_normalize (반복 이모티콘 축약, 이모지는 이미 제거됨)
    normalized = emoticon_normalize(spaced, num_repeats=2)

    # Step 4: LTokenizer 토큰화
    tokens = tokenizer.tokenize(normalized, flatten=True)

    return {
        'emojis': emojis,
        'spaced': spaced,
        'normalized': normalized,
        'tokens': tokens
    }

In [6]:
from soynlp.word import WordExtractor

# content 컬럼의 텍스트를 문장 리스트로 준비
sentences = df['content'].dropna().astype(str).tolist()

# WordExtractor로 단어 점수 학습
word_extractor = WordExtractor(min_frequency=5)
word_extractor.train(sentences)
words = word_extractor.extract()

# LTokenizer 초기화 (cohesion 점수 사용)
cohesion_scores = {word: score.cohesion_forward for word, score in words.items()}
tokenizer = LTokenizer(scores=cohesion_scores)

print(f'추출된 단어 수: {len(cohesion_scores):,}')

training was done. used memory 0.831 Gb
all cohesion probabilities was computed. # words = 1869
all branching entropies was computed # words = 3032
all accessor variety was computed # words = 3032
추출된 단어 수: 1,597


In [7]:
# Sample Testing
sample = df['content'].dropna().iloc[10]
print(f'[원문]\n{sample}\n')

result = preprocess_review(sample, spacing, tokenizer)
print(f'추출 이모지: {result["emojis"]}')
print(f'띄어쓰기 보정: {result["spaced"]}')
print(f'이모티콘 정규화: {result["normalized"]}')
print(f'토큰 리스트: {result["tokens"]}')

[원문]
토스 포인트 받을려고 받으시는분 절대 받지 마세요 포인트 안들어옴 사기 입니다

추출 이모지: []
띄어쓰기 보정: 토스 포인트 받을려고 받으시는 분 절대 받지 마세요 포인트 안 들어옴 사기 입니다
이모티콘 정규화: 토스 포인트 받을려고 받으시는 분 절대 받지 마세요 포인트 안 들어옴 사기 입니다
토큰 리스트: ['토스', '포인트', '받을려고', '받으', '시는', '분', '절대', '받지', '마세요', '포인트', '안', '들어', '옴', '사기', '입니다']


In [8]:
from tqdm.notebook import tqdm
tqdm.pandas()

# 전처리 실행
results = df['content'].progress_apply(
    lambda x: preprocess_review(x, spacing, tokenizer)
)

# 결과를 새 컬럼으로 추가
df['emojis']     = results.apply(lambda x: x['emojis'])
df['spaced']     = results.apply(lambda x: x['spaced'])
df['normalized'] = results.apply(lambda x: x['normalized'])
df['tokens']     = results.apply(lambda x: x['tokens'])

print('전처리 완료!')
df[['content', 'spaced', 'normalized', 'tokens', 'emojis']].head(10)

  0%|          | 0/1919 [00:00<?, ?it/s]

전처리 완료!


,content,spaced,normalized,tokens,emojis
0,굿,굿,굿,[굿],[]
1,할인 쿠폰이 막상 내가 사고싶은 물건엔 적용되지 않음.,할인 쿠폰이 막상 내가 사고 싶은 물건엔 적용되지 않음.,할인 쿠폰이 막상 내가 사고 싶은 물건엔 적용되지 않음.,"[할인, 쿠폰, 이, 막상, 내가, 사고, 싶은, 물건, 엔, 적용, 되지, 않음, .]",[]
2,배송이 빠르고 실시간 배송정보가 알림으로 떠서 아주 만족스럽습니다.,배송이 빠르고 실시간 배송 정보가 알림으로 떠서 아주 만족스럽습니다.,배송이 빠르고 실시간 배송 정보가 알림으로 떠서 아주 만족스럽습니다.,"[배송, 이, 빠르고, 실시간, 배송, 정보, 가, 알림, 으로, 떠서, 아주, 만...",[]
3,쇼핑하기 편리해요 쿠폰 적용도 잘되고 다양한 브랜드가 많이 있어서 쇼핑몰을 가지 않...,쇼핑하기 편리해요 쿠폰 적용도 잘 되고 다양한 브랜드가 많이 있어서 쇼핑몰을 가지 ...,쇼핑하기 편리해요 쿠폰 적용도 잘 되고 다양한 브랜드가 많이 있어서 쇼핑몰을 가지 ...,"[쇼핑, 하기, 편리해요, 쿠폰, 적용, 도, 잘, 되고, 다양한, 브랜드, 가, ...",[]
4,여러스타일의 브랜드를 같이 볼수있어서 너무 좋아요~^^♡,여러 스타일의 브랜드를 같이 볼 수 있어서 너무 좋아요~ ♡,여러 스타일의 브랜드를 같이 볼 수 있어서 너무 좋아요~ ♡,"[여러, 스타일, 의, 브랜드, 를, 같이, 볼, 수, 있어서, 너무, 좋아요, ~...",[]
5,"좋아요,","좋아요,","좋아요,","[좋아요, ,]",[]
6,빠르고 편함,빠르고 편함,빠르고 편함,"[빠르고, 편함]",[]
7,늘 잘사용하고 있어요 ^^ 할인율 높아서 좋아요 ^^,늘 잘 사용하고 있어요 할인율 높아서 좋아요,늘 잘 사용하고 있어요 할인율 높아서 좋아요,"[늘, 잘, 사용하, 고, 있어요, 할인, 율, 높아서, 좋아요]",[]
8,편하게 구경하고 싶은대 데 뭐 자꾸 팝업이 떠. 안 알려줘도 좋으니까 혼자 편하게 ...,편하게 구경하고 싶은 대 데 뭐 자꾸 팝업이 떠. 안 알려줘도 좋으니까 혼자 편하게...,편하게 구경하고 싶은 대 데 뭐 자꾸 팝업이 떠. 안 알려줘도 좋으니까 혼자 편하게...,"[편하게, 구경하, 고, 싶은, 대, 데, 뭐, 자꾸, 팝업, 이, 떠., 안, 알...",[]
9,배송안와서 문의하면 확인 해주는데만 3일~7일 걸리고 품절 되서 환불해준다고 하는 ...,배송안 와서 문의하면 확인 해주는 데만 3일~7일 걸리고 품절 되서 환불해준다고 하...,배송안 와서 문의하면 확인 해주는 데만 3일~7일 걸리고 품절 되서 환불해준다고 하...,"[배송, 안, 와서, 문의, 하면, 확인, 해주, 는, 데만, 3일, ~7일, 걸리...",[]


In [9]:
output_path = '29cm_reviews_preprocessed.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>